# Session 4 - Indexes and Query Optimization Basics

Welcome. In this session we will learn:

- what an **index** is for
- how to read **`EXPLAIN`**
- practical patterns that help the planner

Sample tables are in database **`training`** on **your** MySQL. Use names like `training.customers`.


---
# Chapter 1 - What is an index?

Think of a book index:

- Without it, you scan every page
- With it, you jump to the right page quickly

In MySQL, a **B-tree index** on `customer_id` helps the database find matching orders faster.

Trade-off:

- **Faster reads** for some filters and joins
- **Slower writes** (insert/update/delete must maintain the index)
- Uses **disk space**


**When indexes help**

| Filter style | Typical plan | Why |
|--------------|--------------|-----|
| Selective (`customer_id = 1001`) | Index lookup | few matching rows |
| Broad (`freight > 100`) | May scan table | many rows still match |

**Selective filters** (few matching rows) love indexes.  
**Broad filters** (most rows match) may still scan the table.


---
# Chapter 2 - Indexes already on these tables

Useful indexes were created when the lab database was built (on keys like `customer_id`, `order_date`, and foreign keys).

Instead we **feel** indexes through `EXPLAIN` in the next chapter.


<div style="background-color:#FFF4CC; border:2px solid #E6A800; border-radius:8px; padding:16px 18px; margin:16px 0;">
<h2 style="margin:0 0 10px 0; color:#92400E;">Quiz</h2>
<p style="margin:0 0 10px 0; font-size:1.05em; font-weight:600;">SQL - Session 4 - Indexes</p>
<p style="margin:0 0 10px 0;"><a href="https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s04-indexes-01">https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s04-indexes-01</a></p>
<p style="margin:0; color:#444;">Use the login ID your trainer provided. Every student must attempt this quiz.</p>
</div>


---
# Chapter 3 - Reading EXPLAIN

`EXPLAIN` shows the path MySQL chooses for a query.

For class, use plain `EXPLAIN`.


### Lookup by primary key (expect index/PK scan)


In [1]:
EXPLAIN
SELECT *
FROM training.customers
WHERE customer_id = 1001;


+----+-------------+-----------+------------+-------+---------------+---------+---------+-------+------+----------+-------+
| id | select_type | table     | partitions | type  | possible_keys | key     | key_len | ref   | rows | filtered | Extra |
+----+-------------+-----------+------------+-------+---------------+---------+---------+-------+------+----------+-------+
|  1 | SIMPLE      | customers | NULL       | const | PRIMARY       | PRIMARY | 4       | const |    1 |   100.00 | NULL  |
+----+-------------+-----------+------------+-------+---------------+---------+---------+-------+------+----------+-------+


### Filter on an indexed date column


In [2]:
EXPLAIN
SELECT order_id, customer_id, order_date, status
FROM training.orders
WHERE order_date = DATE '2023-06-15';


+----+-------------+--------+------------+------+-----------------------+-----------------------+---------+-------+------+----------+-------+
| id | select_type | table  | partitions | type | possible_keys         | key                   | key_len | ref   | rows | filtered | Extra |
+----+-------------+--------+------------+------+-----------------------+-----------------------+---------+-------+------+----------+-------+
|  1 | SIMPLE      | orders | NULL       | ref  | idx_orders_order_date | idx_orders_order_date | 3       | const |   57 |   100.00 | NULL  |
+----+-------------+--------+------------+------+-----------------------+-----------------------+---------+-------+------+----------+-------+


### Filter on a non-selective expression (often a seq scan)


In [3]:
EXPLAIN
SELECT order_id, freight
FROM training.orders
WHERE freight > 50;


+----+-------------+--------+------------+------+---------------+------+---------+------+-------+----------+-------------+
| id | select_type | table  | partitions | type | possible_keys | key  | key_len | ref  | rows  | filtered | Extra       |
+----+-------------+--------+------------+------+---------------+------+---------+------+-------+----------+-------------+
|  1 | SIMPLE      | orders | NULL       | ALL  | NULL          | NULL | NULL    | NULL | 38996 |    33.33 | Using where |
+----+-------------+--------+------------+------+---------------+------+---------+------+-------+----------+-------------+


### What to look for in the plan

Plan text differs by engine, but look for the same ideas:

- A **wide scan** of many rows vs a **tight lookup** by key
- Whether an **index** name appears
- How tables are **joined** (nested loops / hash-style joins)

None of these is "always bad". A full scan can be right for small tables or broad filters.


<div style="background-color:#FFF4CC; border:2px solid #E6A800; border-radius:8px; padding:16px 18px; margin:16px 0;">
<h2 style="margin:0 0 10px 0; color:#92400E;">Quiz</h2>
<p style="margin:0 0 10px 0; font-size:1.05em; font-weight:600;">SQL - Session 4 - EXPLAIN</p>
<p style="margin:0 0 10px 0;"><a href="https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s04-explain-01">https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s04-explain-01</a></p>
<p style="margin:0; color:#444;">Use the login ID your trainer provided. Every student must attempt this quiz.</p>
</div>


---
# Chapter 4 - Practical patterns that help

1. Filter on columns that already have indexes when you can.
2. Avoid wrapping indexed columns in functions in `WHERE`  
   (prefer `order_date >= DATE '2023-01-01'` over `EXTRACT(YEAR FROM order_date) = 2023` when possible).
3. Select only the columns you need.
4. Use `LIMIT` while exploring.
5. Join on keys (`customer_id`, `order_id`, ...) - that is how indexes help joins too.


### Sargable date range vs extracting year


In [4]:
EXPLAIN
SELECT COUNT(*)
FROM training.orders
WHERE order_date >= DATE '2023-01-01'
  AND order_date < DATE '2024-01-01';


+----+-------------+--------+------------+-------+-----------------------+-----------------------+---------+------+-------+----------+--------------------------+
| id | select_type | table  | partitions | type  | possible_keys         | key                   | key_len | ref  | rows  | filtered | Extra                    |
+----+-------------+--------+------------+-------+-----------------------+-----------------------+---------+------+-------+----------+--------------------------+
|  1 | SIMPLE      | orders | NULL       | range | idx_orders_order_date | idx_orders_order_date | 3       | NULL | 19498 |   100.00 | Using where; Using index |
+----+-------------+--------+------------+-------+-----------------------+-----------------------+---------+------+-------+----------+--------------------------+


### Compare year extract vs date range in EXPLAIN


In [5]:
EXPLAIN
SELECT COUNT(*)
FROM training.orders
WHERE EXTRACT(YEAR FROM order_date) = 2023;


+----+-------------+--------+------------+-------+---------------+-----------------------+---------+------+-------+----------+--------------------------+
| id | select_type | table  | partitions | type  | possible_keys | key                   | key_len | ref  | rows  | filtered | Extra                    |
+----+-------------+--------+------------+-------+---------------+-----------------------+---------+------+-------+----------+--------------------------+
|  1 | SIMPLE      | orders | NULL       | index | NULL          | idx_orders_order_date | 3       | NULL | 38996 |   100.00 | Using where; Using index |
+----+-------------+--------+------------+-------+---------------+-----------------------+---------+------+-------+----------+--------------------------+


---
# Chapter 5 - Indexes with a TEMPORARY table

Create a small **temporary** copy, add an index, and run `EXPLAIN`.

Temporary tables disappear when the session ends - safe for class (it disappears when the session ends).

> If you re-run this cell in the **same** session and see “already exists”, restart the kernel (or open a new connection) and run it once.


### Temporary table + index + EXPLAIN


In [6]:
CREATE TEMPORARY TABLE temp_orders AS
SELECT order_id, customer_id, order_date, status, freight
FROM training.orders
WHERE order_date >= DATE '2023-01-01'
  AND order_date < DATE '2023-04-01';

CREATE INDEX temp_orders_customer_id_idx
  ON temp_orders (customer_id);

EXPLAIN
SELECT *
FROM temp_orders
WHERE customer_id = 1001;


+----+-------------+-----------------+------------+------+---------------------------------+---------------------------------+---------+-------+------+----------+-------+
| id | select_type | table           | partitions | type | possible_keys                   | key                             | key_len | ref   | rows | filtered | Extra |
+----+-------------+-----------------+------------+------+---------------------------------+---------------------------------+---------+-------+------+----------+-------+
|  1 | SIMPLE      | temp_orders | NULL       | ref  | temp_orders_customer_id_idx | temp_orders_customer_id_idx | 5       | const |    1 |   100.00 | NULL  |
+----+-------------+-----------------+------------+------+---------------------------------+---------------------------------+---------+-------+------+----------+-------+


<div style="background-color:#FFF4CC; border:2px solid #E6A800; border-radius:8px; padding:16px 18px; margin:16px 0;">
<h2 style="margin:0 0 10px 0; color:#92400E;">Quiz</h2>
<p style="margin:0 0 10px 0; font-size:1.05em; font-weight:600;">SQL - Session 4 - Sargable filters</p>
<p style="margin:0 0 10px 0;"><a href="https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s04-sargable-01">https://vmreact.eastus2.cloudapp.azure.com:18094/a/sql-s04-sargable-01</a></p>
<p style="margin:0; color:#444;">Use the login ID your trainer provided. Every student must attempt this quiz.</p>
</div>
